# Benchmark: Transition Rate Computation

This notebook measures the running time of `run_simulations` before the optimization that removes redundant `evaluate` calls and `copy.deepcopy` in `compute_spontaneous_transition_rate` and `compute_mediated_transition_rate`.

In [1]:
import time
import timeit
import numpy as np
import pandas as pd
import cProfile
import pstats
import io

import sys
sys.path.insert(0, "..")

from epydemix.model.epimodel import EpiModel

## Model setup

We benchmark three model configurations:
- **SIR** (simple, 1 age group, spontaneous + mediated transitions)
- **SIR with real population** (16 age groups, mediated transition)
- **SEIR** (more compartments, 1 age group)

In [2]:
def make_sir(population_size=100_000):
    model = EpiModel(
        compartments=["S", "I", "R"],
        parameters={"beta": 0.3, "gamma": 0.1},
        default_population_size=population_size,
    )
    model.add_transition(source="S", target="I", kind="mediated", params=("beta", "I"))
    model.add_transition(source="I", target="R", kind="spontaneous", params="gamma")
    return model


def make_seir(population_size=100_000):
    model = EpiModel(
        compartments=["S", "E", "I", "R"],
        parameters={"beta": 0.3, "sigma": 0.2, "gamma": 0.1},
        default_population_size=population_size,
    )
    model.add_transition(source="S", target="E", kind="mediated", params=("beta", "I"))
    model.add_transition(source="E", target="I", kind="spontaneous", params="sigma")
    model.add_transition(source="I", target="R", kind="spontaneous", params="gamma")
    return model


sir = make_sir()
seir = make_seir()
print(sir)
print(seir)

EpiModel(name='EpiModel')
Compartments: 3
  S, I, R
Transitions: 2
  Transitions between compartments:
    S -> I, params: ('beta', 'I') (kind: mediated)
    I -> R, params: gamma (kind: spontaneous)
Parameters: 2
  Model parameters:
    beta: 0.3
    gamma: 0.1
Population: epydemix_population
  Population size: 100000 individuals
  Demographic groups: 1
    0

EpiModel(name='EpiModel')
Compartments: 4
  S, E, I, R
Transitions: 3
  Transitions between compartments:
    S -> E, params: ('beta', 'I') (kind: mediated)
    E -> I, params: sigma (kind: spontaneous)
    I -> R, params: gamma (kind: spontaneous)
Parameters: 3
  Model parameters:
    beta: 0.3
    sigma: 0.2
    gamma: 0.1
Population: epydemix_population
  Population size: 100000 individuals
  Demographic groups: 1
    0



## Wall-clock timing

We use `timeit` to get stable estimates. `Nsim=50` keeps the run short while giving enough signal.

In [4]:
NSIM = 50
REPEATS = 5
START = "2020-01-01"
END   = "2020-12-31"
RNG   = np.random.default_rng(42)

def bench(model, label):
    times = timeit.repeat(
        lambda: model.run_simulations(
            start_date=START,
            end_date=END,
            Nsim=NSIM,
            rng=np.random.default_rng(42),
        ),
        repeat=REPEATS,
        number=1,
    )
    mean_t = np.mean(times)
    std_t  = np.std(times)
    print(f"{label}: {mean_t:.3f}s ± {std_t:.3f}s  (best {min(times):.3f}s, {REPEATS} repeats, Nsim={NSIM})")
    return times

sir_times  = bench(sir,  "SIR  (1 age group)")
seir_times = bench(seir, "SEIR (1 age group)")

SIR  (1 age group): 1.090s ± 0.039s  (best 1.060s, 5 repeats, Nsim=50)
SEIR (1 age group): 1.614s ± 0.102s  (best 1.545s, 5 repeats, Nsim=50)


## Per-simulation breakdown

In [5]:
for label, times in [("SIR", sir_times), ("SEIR", seir_times)]:
    mean_per_sim = np.mean(times) / NSIM * 1000
    print(f"{label}: {mean_per_sim:.2f} ms / simulation")

SIR: 21.81 ms / simulation
SEIR: 32.27 ms / simulation


## cProfile: where is time spent?

In [6]:
def profile_model(model, label, nsim=20):
    pr = cProfile.Profile()
    pr.enable()
    model.run_simulations(start_date=START, end_date=END, Nsim=nsim, rng=np.random.default_rng(42))
    pr.disable()

    stream = io.StringIO()
    ps = pstats.Stats(pr, stream=stream).sort_stats("cumulative")
    ps.print_stats(20)
    print(f"\n=== {label} ===")
    print(stream.getvalue())

profile_model(sir,  "SIR  profile")
profile_model(seir, "SEIR profile")


=== SIR  profile ===
         351497 function calls (351417 primitive calls) in 0.721 seconds

   Ordered by: cumulative time
   List reduced from 166 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.721    0.721 /Users/nicolo/Desktop/epydemix/epydemix-repo/epydemix/tutorials/../epydemix/model/epimodel.py:728(run_simulations)
       20    0.001    0.000    0.715    0.036 /Users/nicolo/Desktop/epydemix/epydemix-repo/epydemix/tutorials/../epydemix/model/epimodel.py:807(simulate)
       20    0.258    0.013    0.703    0.035 /Users/nicolo/Desktop/epydemix/epydemix-repo/epydemix/tutorials/../epydemix/model/epimodel.py:932(stochastic_simulation)
    10939    0.027    0.000    0.132    0.000 /Users/nicolo/Desktop/epydemix/epydemix-repo/epydemix/tutorials/../epydemix/model/epimodel.py:1020(<listcomp>)
    18726    0.024    0.000    0.113    0.000 /opt/anaconda3/envs/epydemix311/lib/python3.11/site-packages

## Isolate the cost of `evaluate` + `deepcopy`

Direct micro-benchmark of the two hot paths.

In [6]:
import copy
from epydemix.utils.utils import evaluate, create_definitions

T = 365
n_age = 1
params = {"beta": 0.3, "gamma": 0.1}
definitions = create_definitions(params, T, n_age)
t = 100

N_CALLS = 100_000

# Current approach: deepcopy + evaluate
t_current = timeit.timeit(
    lambda: evaluate(expr="gamma", env=copy.deepcopy(definitions))[t],
    number=N_CALLS,
)

# Proposed approach: direct index
t_proposed = timeit.timeit(
    lambda: definitions["gamma"][t],
    number=N_CALLS,
)

print(f"Current  (deepcopy+evaluate): {t_current:.3f}s for {N_CALLS:,} calls  ({t_current/N_CALLS*1e6:.2f} µs/call)")
print(f"Proposed (direct lookup):     {t_proposed:.3f}s for {N_CALLS:,} calls  ({t_proposed/N_CALLS*1e6:.2f} µs/call)")
print(f"Speedup: {t_current/t_proposed:.1f}x")

Current  (deepcopy+evaluate): 2.684s for 100,000 calls  (26.84 µs/call)
Proposed (direct lookup):     0.023s for 100,000 calls  (0.23 µs/call)
Speedup: 116.5x


## Summary table

In [7]:
results = pd.DataFrame({
    "model": ["SIR", "SEIR"],
    "mean_total_s": [np.mean(sir_times), np.mean(seir_times)],
    "std_total_s":  [np.std(sir_times),  np.std(seir_times)],
    "ms_per_sim":   [np.mean(sir_times)/NSIM*1000, np.mean(seir_times)/NSIM*1000],
})
results

,model,mean_total_s,std_total_s,ms_per_sim
0,SIR,1.465214,0.085827,29.304270
1,SEIR,2.063337,0.083960,41.266737


In [ ]:
results = pd.DataFrame({
    "model": ["SIR", "SEIR"],
    "mean_total_s": [np.mean(sir_times), np.mean(seir_times)],
    "std_total_s":  [np.std(sir_times),  np.std(seir_times)],
    "ms_per_sim":   [np.mean(sir_times)/NSIM*1000, np.mean(seir_times)/NSIM*1000],
})
results

,model,mean_total_s,std_total_s,ms_per_sim
0,SIR,1.465214,0.085827,29.304270
1,SEIR,2.063337,0.083960,41.266737


## Micro-benchmark: `compute_mediated_transition_rate` interaction term

Two implementations of the contact matrix interaction:
- **Current**: `np.sum(C * pop[agent] / pop_sizes, axis=1)` — two temporaries, generic ufunc dispatch
- **Proposed**: `C @ (pop[agent] / pop_sizes)` — single matrix-vector product via BLAS


In [10]:
N = 100_000

print("=== Correctness check ===")
for n_age in [1, 4, 16]:
    C = np.random.rand(n_age, n_age)
    pop_agent = np.random.randint(100, 10000, size=n_age).astype(float)
    pop_sizes  = np.random.randint(10000, 100000, size=n_age).astype(float)

    current  = np.sum(C * pop_agent / pop_sizes, axis=1)
    proposed = C @ (pop_agent / pop_sizes)

    assert np.allclose(current, proposed), f"MISMATCH at n_age={n_age}"
    print(f"  n_age={n_age:>2d}: max abs diff = {np.max(np.abs(current - proposed)):.2e}  ✓")

print()
print("=== Timing ===")
for n_age in [1, 4, 16]:
    C = np.random.rand(n_age, n_age)
    pop_agent = np.random.randint(100, 10000, size=n_age).astype(float)
    pop_sizes  = np.random.randint(10000, 100000, size=n_age).astype(float)

    t_current = timeit.timeit(
        lambda: np.sum(C * pop_agent / pop_sizes, axis=1),
        number=N,
    )
    t_proposed = timeit.timeit(
        lambda: C @ (pop_agent / pop_sizes),
        number=N,
    )

    print(f"  n_age={n_age:>2d} | current: {t_current/N*1e6:.3f} µs/call  "
          f"| proposed: {t_proposed/N*1e6:.3f} µs/call  "
          f"| speedup: {t_current/t_proposed:.2f}x")


=== Correctness check ===
  n_age= 1: max abs diff = 0.00e+00  ✓
  n_age= 4: max abs diff = 1.39e-17  ✓
  n_age=16: max abs diff = 2.22e-16  ✓

=== Timing ===
  n_age= 1 | current: 8.190 µs/call  | proposed: 1.924 µs/call  | speedup: 4.26x
  n_age= 4 | current: 7.508 µs/call  | proposed: 1.931 µs/call  | speedup: 3.89x
  n_age=16 | current: 8.136 µs/call  | proposed: 1.991 µs/call  | speedup: 4.09x


## Numba-accelerated `multinomial`

The probability computation inside `multinomial` is a small pure-numeric loop — a good candidate for `@njit`. The strategy:
- compile the probability computation with Numba (no Python overhead, no numpy dispatch)
- keep the `rng.multinomial` draw in Python to preserve Generator seeding/reproducibility


In [9]:
from numba import njit
from epydemix.utils.utils import multinomial

@njit(cache=True)
def _multinomial_numba_core(n, rates, stay_idx, mask, dt, apply_linear_approximation):
    """
    Compiled probability computation.  Returns a float64 probs array.
    The actual multinomial draw is done outside to keep the numpy Generator
    API (and reproducibility) intact.
    """
    k = len(rates)
    probs = np.zeros(k)

    if n <= 0:
        probs[stay_idx] = 1.0
        return probs

    H = 0.0
    for i in range(k):
        if mask[i]:
            H += rates[i] * dt

    if H <= 0.0:
        probs[stay_idx] = 1.0
        return probs

    if apply_linear_approximation:
        for i in range(k):
            if mask[i]:
                probs[i] = rates[i] * dt
        probs[stay_idx] = 1.0 - H
        return probs

    p_leave = -np.expm1(-H)
    scale = p_leave / H
    for i in range(k):
        if mask[i]:
            probs[i] = rates[i] * dt * scale
    probs[stay_idx] = 1.0 - p_leave
    return probs


def multinomial_numba(n, rates, stay_idx, mask, dt,
                      apply_linear_approximation=False, rng=None, **kwargs):
    probs = _multinomial_numba_core(n, rates, stay_idx, mask, dt, apply_linear_approximation)
    if n <= 0:
        out = np.zeros(len(rates), dtype=int)
        out[stay_idx] = int(n)
        return out
    return rng.multinomial(int(n), probs)


# Warm-up JIT compilation (excluded from timing)
print("Warming up JIT...", end=" ")
_n_age = 3
_rates = np.array([0.1, 0.05, 0.0])
_mask  = np.array([False, True, True])
_ = _multinomial_numba_core(100.0, _rates, 0, _mask, 1.0, False)
print("done.")


Warming up JIT... done.


In [10]:
rng = np.random.default_rng(42)
N_CALLS = 100_000

print("=== Correctness check ===")
for C, label in [(3, "SIR  (C=3)"), (4, "SEIR (C=4)"), (17, "C=17 (16 age groups)")]:
    rates    = np.random.rand(C) * 0.2
    mask     = np.ones(C, dtype=bool); mask[0] = False
    stay_idx = 0
    n        = 5000.0
    dt       = 1.0

    current  = multinomial(n, rates, stay_idx, mask, dt, rng=np.random.default_rng(0))
    proposed = multinomial_numba(n, rates, stay_idx, mask, dt, rng=np.random.default_rng(0))
    assert np.array_equal(current, proposed), f"MISMATCH at {label}: {current} vs {proposed}"
    print(f"  {label}: ✓  {current}")

print()
print("=== Timing ===")
for C, label in [(3, "SIR  (C=3)"), (4, "SEIR (C=4)"), (17, "C=17 (16 age groups)")]:
    rates    = np.random.rand(C) * 0.2
    mask     = np.ones(C, dtype=bool); mask[0] = False
    stay_idx = 0
    n        = 5000.0
    dt       = 1.0

    t_current = timeit.timeit(
        lambda: multinomial(n, rates, stay_idx, mask, dt, rng=rng),
        number=N_CALLS,
    )
    t_numba = timeit.timeit(
        lambda: multinomial_numba(n, rates, stay_idx, mask, dt, rng=rng),
        number=N_CALLS,
    )

    print(f"  {label} | current: {t_current/N_CALLS*1e6:.2f} µs  "
          f"| numba: {t_numba/N_CALLS*1e6:.2f} µs  "
          f"| speedup: {t_current/t_numba:.2f}x")


=== Correctness check ===
  SIR  (C=3): ✓  [4023  643  334]
  SEIR (C=4): ✓  [3012  552  643  793]
  C=17 (16 age groups): ✓  [1909  162  521  289  159  256  223   46   89  145  229  349   34   51
  171  217  150]

=== Timing ===
  SIR  (C=3) | current: 12.69 µs  | numba: 3.31 µs  | speedup: 3.84x
  SEIR (C=4) | current: 12.78 µs  | numba: 3.36 µs  | speedup: 3.81x
  C=17 (16 age groups) | current: 12.98 µs  | numba: 4.68 µs  | speedup: 2.77x
